In [1]:
import os
import pandas as pd
from sqlalchemy import create_engine

DB_USER = os.getenv("POSTGRES_USER")
DB_PASS = os.getenv("POSTGRES_PASSWORD")
DB_NAME = os.getenv("POSTGRES_DB")
DB_HOST = os.getenv("POSTGRES_HOST", "localhost")
DB_PORT = os.getenv("POSTGRES_PORT", "5432")

print("USER:", DB_USER)
print("DB:", DB_NAME)
print("HOST:", DB_HOST)
print("PORT:", DB_PORT)

USER: alumno
DB: modelizado
HOST: localhost
PORT: 5432


In [2]:
engine = create_engine(
    f"postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)
print("Conexión establecida")

Conexión establecida


In [3]:
pd.read_sql("SELECT version();", engine)

,version
0,PostgreSQL 16.15 (Debian 16.15-1.pgdg13+2) on ...


In [4]:
query = """
CREATE TABLE IF NOT EXISTS clientes (
    cliente_id SERIAL PRIMARY KEY,
    nombre VARCHAR(50),
    email VARCHAR(100),
    region VARCHAR(50),
    segmento VARCHAR(20),
    estado VARCHAR(20)
);
"""

with engine.connect() as conn:
    conn.execute(query)
    conn.commit()

print("Tabla clientes creada")

ObjectNotExecutableError: Not an executable object: '\nCREATE TABLE IF NOT EXISTS clientes (\n    cliente_id SERIAL PRIMARY KEY,\n    nombre VARCHAR(50),\n    email VARCHAR(100),\n    region VARCHAR(50),\n    segmento VARCHAR(20),\n    estado VARCHAR(20)\n);\n'

In [5]:
from sqlalchemy import text

query = """
CREATE TABLE IF NOT EXISTS clientes (
    cliente_id SERIAL PRIMARY KEY,
    nombre VARCHAR(50),
    email VARCHAR(100),
    region VARCHAR(50),
    segmento VARCHAR(20),
    estado VARCHAR(20)
);
"""

with engine.connect() as conn:
    conn.execute(text(query))
    conn.commit()

print("Tabla clientes creada")

Tabla clientes creada


In [6]:
insert_query = """
INSERT INTO clientes (nombre, email, region, segmento, estado) VALUES
    ('Bruno', 'bruno@test.com', 'Buenos Aires', 'Premium', 'Activo'),
    ('Carla', 'carla@test.com', 'Córdoba', 'Estándar', 'Activo'),
    ('Diego', 'diego@test.com', NULL, 'Premium', 'Inactivo'),
    ('Ana', 'ana@test.com', 'Santa Fe', 'Estándar', 'Activo'),
    ('Elena', 'elena@test.com', 'Buenos Aires', 'Premium', 'Activo');
"""

with engine.connect() as conn:
    conn.execute(text(insert_query))
    conn.commit()

print("Datos insertados")

Datos insertados


In [7]:
df = pd.read_sql("SELECT * FROM clientes;", engine)
df

,cliente_id,nombre,email,region,segmento,estado
0,1,Bruno,bruno@test.com,Buenos Aires,Premium,Activo
1,2,Carla,carla@test.com,Córdoba,Estándar,Activo
2,3,Diego,diego@test.com,NaN,Premium,Inactivo
3,4,Ana,ana@test.com,Santa Fe,Estándar,Activo
4,5,Elena,elena@test.com,Buenos Aires,Premium,Activo


In [8]:
df.isnull().sum()

cliente_id    0
nombre        0
email         0
region        1
segmento      0
estado        0
dtype: int64

In [9]:
(df.isnull().sum() / len(df)) * 100

cliente_id     0.0
nombre         0.0
email          0.0
region        20.0
segmento       0.0
estado         0.0
dtype: float64

In [10]:
df.duplicated().sum()

np.int64(0)

In [11]:
df.nunique()

cliente_id    5
nombre        5
email         5
region        3
segmento      2
estado        2
dtype: int64

In [12]:
df["region"].value_counts(dropna=False)

region
Buenos Aires    2
Córdoba         1
NaN             1
Santa Fe        1
Name: count, dtype: int64

In [13]:
df["segmento"].value_counts()

segmento
Premium     3
Estándar    2
Name: count, dtype: int64

In [14]:
df["estado"].value_counts()

estado
Activo      4
Inactivo    1
Name: count, dtype: int64

In [15]:
# Regla 1: cliente_id no nulo
print("Regla 1 - cliente_id nulos:", df["cliente_id"].isnull().sum())

# Regla 2: cliente_id único
print("Regla 2 - cliente_id duplicados:", df["cliente_id"].duplicated().sum())

# Regla 3: email único
print("Regla 3 - email duplicados:", df["email"].duplicated().sum())

# Regla 4: region completa
print("Regla 4 - region nulos:", df["region"].isnull().sum())

Regla 1 - cliente_id nulos: 0
Regla 2 - cliente_id duplicados: 0
Regla 3 - email duplicados: 0
Regla 4 - region nulos: 1
